In [ ]:
import deeplay as dl
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch_geometric.loader import DataLoader
from torchvision.transforms import Compose

from trajan.data import TracksDataFrame
from trajan.dataset import GraphDataset
from trajan.graph import GraphFromTrajectories
from trajan.transforms import RandomFlip, RandomRotation

In [ ]:
Dt = 50
max_frame_distance = 3

In [ ]:
#data = pd.read_csv("data/simulated/fbm_dataset.csv")
#data = particle_tracking.TracksDataFrame(data)

data = pd.read_csv("../data/cytoplasmic_data/tracks.csv", skiprows=1)
data = TracksDataFrame(data, frame_rate=10)

data_description = data.describe_tracks()
display_labels = data_description['particle_types']

In [ ]:
train_data, val_data = data.split_train_test()

graph_builder, position_scale = GraphFromTrajectories.from_tracks(train_data, Dt, max_frame_distance)

train_graphs = graph_builder(train_data, target_column="type", split_tracks=True)
val_graphs = graph_builder(val_data, target_column="type", split_tracks=True)

In [ ]:
train_dataset_size = 2 * len(train_graphs)
val_dataset_size = 2 * len(val_graphs)

batch_size = 16

transform = Compose([
        RandomRotation(),
        RandomFlip(),
        ])

train_dataset = GraphDataset(
    train_graphs,
    Dt,
    train_dataset_size,
    position_scale=position_scale,
    transform=transform,
    target="global"
)
val_dataset = GraphDataset(
    val_graphs,
    Dt,
    train_dataset_size,
    position_scale=position_scale,
    transform=transform,
    target="global"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    drop_last=True,
    num_workers=0,
)


In [ ]:
max_frame_distance = 3
Dt = 50
lr = 5e-4
wd = 1e-5
encoder_dimension = 96
num_blocks = 3
batch_size = 16
num_epochs = 10
#class_weights = torch.tensor([1.89, 5.56, 3.45])
sample_balanced = True

magik = dl.GraphToGlobalMPM(
    [encoder_dimension] * num_blocks,
    out_activation=nn.Softmax(dim=1),
    out_features=3,
).create()

classifier_magik = dl.CategoricalClassifier(
    model=magik,
    optimizer=dl.Adam(lr=lr, weight_decay=wd),
    loss=nn.CrossEntropyLoss(),
    num_classes=3,
).build()

trainer_magik = dl.Trainer(max_epochs=num_epochs, accelerator="auto")

trainer_magik.fit(classifier_magik, train_loader, val_loader)

In [ ]:
trainer_magik.history.plot()

truth, preds = [], []

for batch in val_loader:
    y_true = batch.y
    y_pred = torch.argmax(classifier_magik(batch), dim=1)
    truth.append(y_true)
    preds.append(y_pred)

truth, preds = torch.concat(truth).numpy(), torch.concat(preds).numpy()

cm = confusion_matrix(truth, preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                            display_labels=display_labels)

disp.plot()

report = classification_report(truth, preds, target_names=display_labels, output_dict=True)
report_df = pd.DataFrame(report).T.round(2)